# Serbia RAG Retrieval Simulation (English Queries)

This notebook simulates retrieval on the **current live chunk index** using the repo's `ServiceContainer` and `RetrievalService`.

What it covers:
- load the active backend config
- inspect indexed sources and municipality coverage
- run English smoke queries against Serbia municipal and national sources
- compare retrieval modes (`semantic`, `hybrid`, `lexical`)


## Notebook Setup (VS Code / Cursor)

1. Create and activate the project environment.
```bash
conda activate wb-ldt-app-backend
```
2. Install dependencies used by this notebook.
```bash
pip install -e .
pip install jupyter ipykernel python-dotenv
```
3. Register the kernel once (if not already present).
```bash
python -m ipykernel install --user --name wb-ldt-app-backend --display-name "Python (wb-ldt-app-backend)"
```
4. Create a local env file at repo root: `.env.notebook`.
```bash
LDT_STORAGE_BACKEND=postgres
LDT_DATABASE_URL=<SUPABASE_SESSION_POOLER_URI>
LDT_EMBEDDING_PROVIDER=openai
LDT_OPENAI_API_KEY=<OPENAI_API_KEY>
LDT_EMBEDDING_MODEL=text-embedding-3-large
LDT_EMBEDDING_DIMENSIONS=1536
LDT_DOCUMENT_STORE_BACKEND=local
LDT_AUTO_SEED_SOURCES=false
```
5. If you intentionally set `LDT_DOCUMENT_STORE_BACKEND=gcs`, also configure Google ADC.
```bash
export GOOGLE_APPLICATION_CREDENTIALS="/absolute/path/to/service-account.json"
```
6. In VS Code or Cursor, open this notebook and select kernel `Python (wb-ldt-app-backend)`.
7. Run cells top-to-bottom. Re-run the live monitor cells while Stage 3 is running.

Troubleshooting:
- If you see `storage_backend: memory`, your env file was not loaded.
- If you see OpenAI key errors, verify `LDT_OPENAI_API_KEY` in `.env.notebook`.
- If Postgres connection fails, use the exact **Session pooler** URI from Supabase (do not hand-type hostnames).


In [1]:
import os, sys
from pathlib import Path

# Resolve repo root robustly (works whether notebook is opened from repo root or /notebooks).
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if ROOT is None:
    raise RuntimeError("Could not locate repo root containing /src")
os.chdir(ROOT)

# Load environment files early, before any DB/settings checks.
loaded_env_files = []
try:
    from dotenv import load_dotenv
    for env_name, override in ((".env.notebook", True), (".env", False)):
        env_path = ROOT / env_name
        if env_path.exists():
            load_dotenv(env_path, override=override)
            loaded_env_files.append(str(env_path))
except Exception as exc:
    print(f"python-dotenv unavailable or failed to load: {exc}")

print(f"Python executable: {sys.executable}")
print(f"Repo root: {ROOT}")
print("Loaded env files:")
if loaded_env_files:
    for path in loaded_env_files:
        print(f"  - {path}")
else:
    print("  - none")

print("Resolved env snapshot:")
print("  LDT_STORAGE_BACKEND=", os.getenv("LDT_STORAGE_BACKEND"))
print("  LDT_EMBEDDING_PROVIDER=", os.getenv("LDT_EMBEDDING_PROVIDER"))
print("  LDT_EMBEDDING_DIMENSIONS=", os.getenv("LDT_EMBEDDING_DIMENSIONS"))
print("  LDT_DATABASE_URL set=", bool(os.getenv("LDT_DATABASE_URL")))
print("  LDT_OPENAI_API_KEY set=", bool(os.getenv("LDT_OPENAI_API_KEY")))


Python executable: /Users/sonle/opt/anaconda3/envs/wb-ldt-app-backend/bin/python
Repo root: /Users/sonle/Documents/work/WB/wb-ldt-app
Loaded env files:
  - /Users/sonle/Documents/work/WB/wb-ldt-app/.env.notebook
Resolved env snapshot:
  LDT_STORAGE_BACKEND= postgres
  LDT_EMBEDDING_PROVIDER= openai
  LDT_EMBEDDING_DIMENSIONS= 1536
  LDT_DATABASE_URL set= True
  LDT_OPENAI_API_KEY set= True


In [2]:
import os, psycopg
database_url = os.getenv("LDT_DATABASE_URL")
if not database_url:
    raise RuntimeError("LDT_DATABASE_URL is empty. Ensure .env.notebook exists at repo root and rerun Cell 1.")

with psycopg.connect(database_url) as c:
    with c.cursor() as cur:
        cur.execute("select 1")
        print(cur.fetchone())

(1,)


In [3]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path

# Ensure repo root is importable when notebook is opened from /notebooks.
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    ROOT = cwd
elif (cwd.parent / "src").exists():
    ROOT = cwd.parent
else:
    raise RuntimeError("Could not find repo root containing /src")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print(f"Using repo root: {ROOT}")

# Optional: load .env.notebook first, then .env as fallback.
try:
    from dotenv import load_dotenv
    loaded = []
    notebook_env = ROOT / ".env.notebook"
    default_env = ROOT / ".env"
    if notebook_env.exists():
        load_dotenv(notebook_env, override=True)
        loaded.append(notebook_env.name)
    if default_env.exists():
        load_dotenv(default_env, override=False)
        loaded.append(default_env.name)
    if loaded:
        print(f"Loaded env files: {', '.join(loaded)}")
    else:
        print("No .env.notebook or .env file found; relying on current shell env")
except Exception:
    print("python-dotenv not available; relying on existing environment variables")


Using repo root: /Users/sonle/Documents/work/WB/wb-ldt-app
Loaded env files: .env.notebook


In [4]:
from src.config.settings import get_settings

# Refresh settings cache in case environment changed before running this notebook.
get_settings.cache_clear()
settings = get_settings()

config_snapshot = {
    "storage_backend": settings.storage_backend,
    "embedding_provider": settings.embedding_provider,
    "embedding_model": settings.embedding_model,
    "embedding_dimensions": settings.embedding_dimensions,
    "document_store_backend": settings.document_store_backend,
}
print(json.dumps(config_snapshot, indent=2))

if settings.storage_backend.lower() != "postgres":
    raise ValueError("This simulation expects LDT_STORAGE_BACKEND=postgres.")
if not settings.database_url:
    raise ValueError("LDT_DATABASE_URL is required for this simulation.")
if settings.embedding_provider.lower() == "openai" and not settings.openai_api_key:
    raise ValueError("LDT_OPENAI_API_KEY is required when embedding_provider=openai.")


{
  "storage_backend": "postgres",
  "embedding_provider": "openai",
  "embedding_model": "text-embedding-3-large",
  "embedding_dimensions": 1536,
  "document_store_backend": "local"
}


In [5]:
from src.core.container import ServiceContainer

container = ServiceContainer(settings=settings)
repo = container.source_repository
retrieval = container.retrieval_service

print("ServiceContainer initialized")


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
ServiceContainer initialized


In [6]:
# Corpus overview
sources = repo.list_sources()
source_type_counts = Counter(str(s.source_type) for s in sources)
municipality_counts = Counter(s.municipality_id for s in sources if s.municipality_id)

print(f"Total sources: {len(sources)}")
print("Source types:")
for k, v in sorted(source_type_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {k}: {v}")

print("\nTop municipalities by source count:")
for municipality_id, count in municipality_counts.most_common(10):
    print(f"  - {municipality_id}: {count}")


Total sources: 63
Source types:
  - municipal_development_plan: 63

Top municipalities by source count:
  - srb-cacak: 1
  - srb-sabac: 1
  - srb-arilje: 1
  - srb-backa-palanka: 1
  - srb-backi-petrovac: 1
  - srb-bac: 1
  - srb-bajina-basta: 1
  - srb-becej: 1
  - srb-bela-crkva: 1
  - srb-beograd-barajevo: 1


## Live Chunk Ingestion Monitor

Run the next cells repeatedly while Stage 3 is running to confirm chunks are being added and retrieval quality is improving.

In [8]:
def get_chunk_progress_snapshot(*, recent_sources: int = 8):
    with psycopg.connect(settings.database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT COUNT(*) AS total_chunks,
                       COUNT(DISTINCT source_id) AS sources_with_chunks
                FROM source_chunks
                """
            )
            total_chunks, sources_with_chunks = cur.fetchone()

            cur.execute(
                """
                SELECT source_type,
                       COUNT(*) AS chunk_count,
                       COUNT(DISTINCT source_id) AS source_count
                FROM source_chunks
                GROUP BY source_type
                ORDER BY chunk_count DESC, source_type ASC
                """
            )
            by_type = cur.fetchall()

            cur.execute(
                """
                SELECT
                    s.source_id,
                    s.source_payload->>'title' AS title,
                    s.source_payload->>'source_type' AS source_type,
                    s.source_payload->>'registered_at' AS registered_at,
                    COUNT(sc.chunk_id) AS chunk_count
                FROM sources s
                LEFT JOIN source_chunks sc ON sc.source_id = s.source_id
                GROUP BY s.source_id, s.source_payload
                ORDER BY (s.source_payload->>'registered_at')::timestamptz DESC NULLS LAST
                LIMIT %s
                """,
                (recent_sources,),
            )
            recent = cur.fetchall()

    return {
        "total_chunks": int(total_chunks or 0),
        "sources_with_chunks": int(sources_with_chunks or 0),
        "by_type": by_type,
        "recent_sources": recent,
    }


def print_chunk_progress(snapshot: dict):
    print(f"Total chunks: {snapshot['total_chunks']}")
    print(f"Sources with chunks: {snapshot['sources_with_chunks']}")
    print("Chunks by source_type:")
    for source_type, chunk_count, source_count in snapshot["by_type"]:
        print(f"  - {source_type}: chunks={chunk_count}, sources={source_count}")
    print("Most recently registered sources (with current chunk counts):")
    for source_id, title, source_type, registered_at, chunk_count in snapshot["recent_sources"]:
        print(f"  - {registered_at} | {source_type} | chunks={chunk_count}")
        print(f"    {source_id} :: {title}")


In [9]:
chunk_progress = get_chunk_progress_snapshot(recent_sources=12)
print_chunk_progress(chunk_progress)


Total chunks: 3510
Sources with chunks: 4
Chunks by source_type:
  - municipal_development_plan: chunks=3510, sources=4
Most recently registered sources (with current chunk counts):
  - 2026-04-22T16:14:52.319531Z | municipal_development_plan | chunks=0
    serbia-serbia_municipal_development_plans-municipal_development_plan-be-ej-local-development-plan-61cdebb6c5 :: Bečej Local Development Plan
  - 2026-04-22T16:11:39.730109Z | municipal_development_plan | chunks=1734
    serbia-serbia_municipal_development_plans-municipal_development_plan-bogati-local-development-plan-c78e7a0724 :: Bogatić Local Development Plan
  - 2026-04-22T16:10:08.665135Z | municipal_development_plan | chunks=369
    serbia-serbia_municipal_development_plans-municipal_development_plan-gornji-milanovac-local-development-plan-7b7daa49f3 :: Gornji Milanovac Local Development Plan
  - 2026-04-22T16:07:43.864968Z | municipal_development_plan | chunks=731
    serbia-serbia_municipal_development_plans-municipal_develop

In [10]:
def run_search(*, query: str, mode: str, top_k: int = 5, municipality_id: str | None = None, source_types: set[str] | None = None):
    response = retrieval.search(
        query=query,
        mode=mode,
        top_k=top_k,
        municipality_id=municipality_id,
        category=None,
        source_types=source_types,
    )
    rows = []
    for r in response.results:
        rows.append(
            {
                "score": round(r.score, 4),
                "lexical": round(r.lexical_score, 4),
                "semantic": round(r.semantic_score, 4),
                "source_id": r.source_id,
                "title": r.citation_title,
                "snippet": " ".join((r.snippet or "").split())[:220],
            }
        )
    return {
        "query": query,
        "mode": mode,
        "municipality_id": municipality_id,
        "total_results": response.total_results,
        "diagnostics": response.diagnostics,
        "rows": rows,
    }

def print_result_block(label: str, payload: dict):
    print(f"\n=== {label} ===")
    print(f"mode={payload['mode']} | total_results={payload['total_results']} | municipality_id={payload['municipality_id']}")
    print(f"query: {payload['query']}")
    for idx, row in enumerate(payload["rows"], start=1):
        print(f"  {idx}. score={row['score']} lexical={row['lexical']} semantic={row['semantic']}")
        print(f"     title={row['title']}")
        print(f"     source_id={row['source_id']}")
        print(f"     snippet={row['snippet']}")


In [11]:
municipality_counts

Counter({'srb-becej': 1,
         'srb-bogatic': 1,
         'srb-bojnik': 1,
         'srb-gornji-milanovac': 1,
         'srb-ivanjica': 1})

In [14]:
# Pick one municipality for municipal tests.
selected_municipality_id = municipality_counts.most_common(1)[0][0] if municipality_counts else None
selected_municipality_id = 'srb-bogatic'
# selected_municipality_id = 'srb-veliko-gradiste'
print(f"Selected municipality for municipal tests: {selected_municipality_id}")

english_smoke_queries = {
    "municipal": "What are the main development priorities and planned investments in this municipality's local development plan?",
    "municipal_alt": "Which economic development and public infrastructure actions are emphasized in the local development strategy?",
    "national": "What are Serbia's national policy priorities for sustainable development, environment, and public investment?",
}

municipal_source_types = {"municipal_development_plan"}
national_source_types = {"policy_document"}


Selected municipality for municipal tests: srb-bogatic


In [15]:
# Main smoke test in semantic mode (best signal for cross-lingual English->Serbian retrieval).
if selected_municipality_id:
    municipal_res = run_search(
        query=english_smoke_queries["municipal"],
        mode="semantic",
        municipality_id=selected_municipality_id,
        source_types=municipal_source_types,
        top_k=5,
    )
    print_result_block("Municipal Semantic Smoke", municipal_res)

# national_res = run_search(
#     query=english_smoke_queries["national"],
#     mode="semantic",
#     municipality_id=None,
#     source_types=national_source_types,
#     top_k=5,
# )
# print_result_block("National Semantic Smoke", national_res)



=== Municipal Semantic Smoke ===
mode=semantic | total_results=5 | municipality_id=srb-bogatic
query: What are the main development priorities and planned investments in this municipality's local development plan?
  1. score=0.5787 lexical=0.0 semantic=0.5787
     title=Bogatić Local Development Plan
     source_id=serbia-serbia_municipal_development_plans-municipal_development_plan-bogati-local-development-plan-c78e7a0724
     snippet=Document Title: Bogatić Local Development Plan Source Type: municipal_development_plan Section Path: ДРУШТВЕНИ РАЗВОЈ | Приоритетни циљ: Повећање обухвата деце системом предшколског васпитања и образовања Показатељ учинк
  2. score=0.5665 lexical=0.0 semantic=0.5665
     title=Bogatić Local Development Plan
     source_id=serbia-serbia_municipal_development_plans-municipal_development_plan-bogati-local-development-plan-c78e7a0724
     snippet=Document Title: Bogatić Local Development Plan Source Type: municipal_development_plan Section Path: Преглед раз

In [ ]:
import time


def watch_chunk_growth(*, iterations: int = 6, sleep_seconds: int = 30):
    previous_total = None
    for i in range(iterations):
        snapshot = get_chunk_progress_snapshot(recent_sources=5)
        current_total = snapshot["total_chunks"]
        delta = 0 if previous_total is None else current_total - previous_total
        print(f"\n[{i + 1}/{iterations}] total_chunks={current_total} (delta={delta:+d})")
        print_chunk_progress(snapshot)
        if i < iterations - 1:
            time.sleep(sleep_seconds)
        previous_total = current_total


In [ ]:
# Optional live watch while Stage 3 is still ingesting.
# Example: watch_chunk_growth(iterations=12, sleep_seconds=30)
watch_chunk_growth(iterations=3, sleep_seconds=15)


In [13]:
# Optional mode comparison for one municipal query.
# Note: hybrid can be slower on larger indexes because lexical retrieval scans chunk text.
if selected_municipality_id:
    for mode in ["semantic", "hybrid", "lexical"]:
        res = run_search(
            query=english_smoke_queries["municipal_alt"],
            mode=mode,
            municipality_id=selected_municipality_id,
            source_types=municipal_source_types,
            top_k=5,
        )
        print_result_block(f"Municipal Mode Comparison: {mode}", res)



=== Municipal Mode Comparison: semantic ===
mode=semantic | total_results=1 | municipality_id=srb-veliko-gradiste
query: Which economic development and public infrastructure actions are emphasized in the local development strategy?
  1. score=0.3334 lexical=0.0 semantic=0.3334
     title=Veliko Gradište Local Development Plan
     source_id=serbia-serbia_municipal_development_plans-municipal_development_plan-veliko-gradi-te-local-development-plan-cbfeae1fd4
     snippet=Document Title: Veliko Gradište Local Development Plan
Source Type: municipal_development_plan

Unsupported parser for veliko-gradi-te-local-development-plan__pdf. Binary size: 152

=== Municipal Mode Comparison: hybrid ===
mode=hybrid | total_results=1 | municipality_id=srb-veliko-gradiste
query: Which economic development and public infrastructure actions are emphasized in the local development strategy?
  1. score=0.0328 lexical=0.1538 semantic=0.3334
     title=Veliko Gradište Local Development Plan
     source_id=

## Interpretation Checklist

- If `semantic` returns relevant Serbian municipal/national titles for English queries, cross-lingual retrieval is working.
- If `lexical` underperforms while `semantic` works, that is expected for English queries over Serbian text.
- If all modes return zero, check filters (`source_types`, `municipality_id`) and environment (`LDT_STORAGE_BACKEND`, `LDT_DATABASE_URL`).
